In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.multioutput import MultiOutputClassifier
import mlflow
from pathlib import Path
import time

# ==========================================
# 1. SETUP PATH & MLFLOW
# ==========================================
root_path = Path.cwd().parent
mlflow.set_tracking_uri((root_path / "mlruns").as_uri())
mlflow.set_experiment("06_MFO_Feature_Selection")

print("⏳ 1. Memuat Data Training yang Sudah Seimbang (Balanced)...")
# Membaca data hasil tahap 05
train_df = pd.read_csv(root_path / "Data/processed/train_balanced_multilabel.csv")

target_cols = ['risk_depression', 'risk_anxiety', 'risk_stress']
X_train = train_df.drop(columns=target_cols)
Y_train = train_df[target_cols]

# ==========================================
# 2. DEFINISI ALGORITMA MFO BINER
# ==========================================
class BinaryMFO:
    def __init__(self, num_moths, max_iterations, num_features):
        self.num_moths = num_moths
        self.max_iterations = max_iterations
        self.num_features = num_features
        # Inisialisasi posisi ngengat secara acak (0 atau 1)
        self.moths = np.random.randint(2, size=(num_moths, num_features))
        self.flames = np.zeros((num_moths, num_features))
        self.moth_fitness = np.zeros(num_moths)
        self.flame_fitness = np.full(num_moths, -np.inf) # Negatif tak hingga karena kita cari max (F1-Score)
        
        # Base model XGBoost untuk Multi-label
        base_xgb = xgb.XGBClassifier(eval_metric='logloss', use_label_encoder=False, n_jobs=-1, random_state=42)
        self.model = MultiOutputClassifier(base_xgb)

    def evaluate_fitness(self, moth_position):
        # Konversi array posisi ke boolean mask
        selected_features_mask = moth_position == 1
        
        # Jika tidak ada fitur yang dipilih, berikan penalti (fitness sangat rendah)
        if not np.any(selected_features_mask):
            return 0.0 
        
        X_subset = X_train.iloc[:, selected_features_mask]
        
        # Cross-validation dengan F1-Macro untuk melihat performa kombinasi fitur ini
        # (Menggunakan cv=3 agar proses optimasi tidak terlalu lama)
        scores = cross_val_score(self.model, X_subset, Y_train, cv=3, scoring='f1_macro')
        return scores.mean()

    def optimize(self):
        print(f"🚀 Memulai MFO dengan {self.num_moths} Ngengat selama {self.max_iterations} Iterasi...")
        start_time = time.time()
        
        for iteration in range(self.max_iterations):
            print(f"--- Iterasi {iteration + 1}/{self.max_iterations} ---")
            
            # Hitung fitness untuk setiap ngengat
            for i in range(self.num_moths):
                self.moth_fitness[i] = self.evaluate_fitness(self.moths[i])
            
            # Mengurutkan ngengat berdasarkan fitness terbaik
            sorted_indices = np.argsort(self.moth_fitness)[::-1] # Descending (Max F1)
            self.moths = self.moths[sorted_indices]
            self.moth_fitness = self.moth_fitness[sorted_indices]
            
            # Update Flames (Api) - Pada iterasi pertama, ngengat terbaik menjadi api
            if iteration == 0:
                self.flames = np.copy(self.moths)
                self.flame_fitness = np.copy(self.moth_fitness)
            else:
                # Menggabungkan populasi api lama dan ngengat baru, lalu ambil yang terbaik
                combined_population = np.vstack((self.flames, self.moths))
                combined_fitness = np.concatenate((self.flame_fitness, self.moth_fitness))
                
                best_indices = np.argsort(combined_fitness)[::-1][:self.num_moths]
                self.flames = combined_population[best_indices]
                self.flame_fitness = combined_fitness[best_indices]
            
            # Update Posisi Ngengat berdasarkan persamaan Logarithmic Spiral
            a = -1 + iteration * ((-1) / self.max_iterations) # Konversi linier a dari -1 ke -2
            
            for i in range(self.num_moths):
                for j in range(self.num_features):
                    # Tentukan api tujuan (ngengat ke-i menuju api ke-i, dst)
                    flame_no = i if i < self.num_moths else self.num_moths - 1
                    
                    distance_to_flame = abs(self.flames[flame_no, j] - self.moths[i, j])
                    b = 1 # Konstanta bentuk spiral
                    t = (a - 1) * np.random.rand() + 1
                    
                    # Persamaan pembaruan S(Mi, Fj)
                    new_position_continuous = distance_to_flame * np.exp(b * t) * np.cos(t * 2 * np.pi) + self.flames[flame_no, j]
                    
                    # Binarization (Sigmoid function) untuk menentukan dipakai (1) atau dibuang (0)
                    sigmoid_val = 1 / (1 + np.exp(-new_position_continuous))
                    self.moths[i, j] = 1 if sigmoid_val > 0.5 else 0
            
            print(f"✅ F1-Score Terbaik Sementara: {self.flame_fitness[0]:.4f}")
            
        execution_time = time.time() - start_time
        print(f"\n🎉 Optimasi Selesai dalam {execution_time/60:.2f} menit!")
        
        # Kembalikan mask dari fitur terbaik
        best_feature_mask = self.flames[0] == 1
        return best_feature_mask, self.flame_fitness[0]

# ==========================================
# 3. EKSEKUSI & MLFLOW LOGGING
# ==========================================
with mlflow.start_run(run_name="MFO_Feature_Selection"):
    # Parameter MFO (Untuk skripsi, iterasi bisa diperbesar, misal: moths=15, iter=20)
    # Di sini diset kecil agar uji coba di laptop tidak memakan waktu berjam-jam
    NUM_MOTHS = 15
    MAX_ITER = 20 
    
    mfo = BinaryMFO(num_moths=NUM_MOTHS, max_iterations=MAX_ITER, num_features=X_train.shape[1])
    best_mask, best_score = mfo.optimize()
    
    # Menerapkan seleksi pada dataset
    selected_features = X_train.columns[best_mask].tolist()
    X_train_selected = X_train[selected_features]
    
    print("\n" + "="*50)
    print(f"🔥 HASIL SELEKSI FITUR MFO 🔥")
    print(f"Dari {X_train.shape[1]} fitur, MFO mempertahankan {len(selected_features)} fitur inti.")
    print(f"Fitur yang terpilih: {selected_features}")
    print(f"F1-Score Validasi Internal: {best_score:.4f}")
    print("="*50)
    
    # Simpan dataset hasil seleksi
    output_path = root_path / "Data/processed/train_selected_features.csv"
    X_train_selected_full = pd.concat([X_train_selected, Y_train], axis=1)
    X_train_selected_full.to_csv(output_path, index=False)
    
    # Logging ke MLflow
    mlflow.log_param("mfo_num_moths", NUM_MOTHS)
    mlflow.log_param("mfo_iterations", MAX_ITER)
    mlflow.log_metric("mfo_best_cv_f1", best_score)
    mlflow.log_metric("num_features_selected", len(selected_features))
    mlflow.log_text(", ".join(selected_features), "selected_features.txt")

⏳ 1. Memuat Data Training yang Sudah Seimbang (Balanced)...
🚀 Memulai MFO dengan 15 Ngengat selama 20 Iterasi...
--- Iterasi 1/20 ---


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:15:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:15:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:15:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:15:43] WARNING: C:\actions-r

✅ F1-Score Terbaik Sementara: 0.4696
--- Iterasi 2/20 ---


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:16:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:16:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:16:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:16:04] WARNING: C:\actions-r

✅ F1-Score Terbaik Sementara: 0.4696
--- Iterasi 3/20 ---


...

✅ F1-Score Terbaik Sementara: 0.4800
--- Iterasi 19/20 ---


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:24:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:24:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:24:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:24:16] WARNING: C:\actions-r

✅ F1-Score Terbaik Sementara: 0.4800
--- Iterasi 20/20 ---


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:24:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:24:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:24:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [17:24:49] WARNING: C:\actions-r

✅ F1-Score Terbaik Sementara: 0.4800

🎉 Optimasi Selesai dalam 9.58 menit!

🔥 HASIL SELEKSI FITUR MFO 🔥
Dari 37 fitur, MFO mempertahankan 35 fitur inti.
Fitur yang terpilih: ['TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 'VCL1', 'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL11', 'VCL12', 'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education', 'urban', 'gender', 'engnat', 'age', 'religion', 'orientation', 'race', 'voted', 'married']
F1-Score Validasi Internal: 0.4800
